In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)


PyTorch: 2.9.0+cu126
CUDA available: True
Device: cuda


In [3]:
df_train = pd.read_csv("train.csv", parse_dates=["date"])
df_test = pd.read_csv("test.csv", parse_dates=["date"])
df_stores = pd.read_csv("stores.csv")
df_oil = pd.read_csv("oil.csv", parse_dates=["date"])
df_holidays = pd.read_csv("holidays_events.csv", parse_dates=["date"])

In [ ]:
# 檢查油價資料的缺漏值 

# 設定日期為索引
df_oil = df_oil.set_index('date')

# 包含 df_train_mod (訓練集) 和 df_test (測試集) 的所有日期
start_date = df_train['date'].min()
end_date = df_test['date'].max()
# 日期範圍
full_date_range = pd.date_range(start=start_date, end=end_date, freq='D')
print(f"油價資料處理範圍: {start_date.date()} 到 {end_date.date()}")

# 重設索引
df_oil = df_oil.reindex(full_date_range)

# 索引重新命名為 date
df_oil.index.name = 'date'

# limit_direction='both' 確保頭尾的缺值能被處理
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].interpolate(method='linear', limit_direction='both')

# 重設索引
df_oil = df_oil.reset_index()

# 檢查是否還有 NaN 
print(f"缺漏值數量 (處理後): {df_oil['dcoilwtico'].isna().sum()}")
df_oil


油價資料處理範圍: 2013-01-01 到 2017-08-31
缺漏值數量 (處理後): 0


,date,dcoilwtico
0,2013-01-01,93.140000
1,2013-01-02,93.140000
2,2013-01-03,92.970000
3,2013-01-04,93.120000
4,2013-01-05,93.146667
...,...,...
1699,2017-08-27,46.816667
1700,2017-08-28,46.400000
1701,2017-08-29,46.460000
1702,2017-08-30,45.960000


In [ ]:
# 清洗假日資料 

# 紀錄被轉移的內容
condition_transferred = df_holidays['transferred'] == True

# 移除補班日 (type == 'Work Day') 這些日子雖然在表裡，但其實是要上班的
condition_workday = df_holidays['type'] == 'Work Day'

# 保留 "既沒有被轉移" 且 "不是補班日" 的資料
df_holidays_clean = df_holidays[~condition_transferred & ~condition_workday].copy()

# 需要保留日期、類型、地區資訊
df_holidays_clean = df_holidays_clean[['date', 'type', 'locale', 'locale_name', 'description']]
print(f"清洗後假日資料筆數: {df_holidays_clean.shape[0]}")
print("-" * 30)
print("清洗後範例 (前5筆):")
display(df_holidays_clean.head())

# 檢查是否還有 Work Day 或 transferred == True
check_errors = df_holidays_clean[
    (df_holidays_clean['type'] == 'Work Day') |
    (df_holidays['transferred'] == True) 
]
print(check_errors)

清洗後假日資料筆數: 333
------------------------------
清洗後範例 (前5筆):


,date,type,locale,locale_name,description
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba


Empty DataFrame
Columns: [date, type, locale, locale_name, description]
Index: []


C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_24660\234136613.py:21: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  check_errors = df_holidays_clean[


In [ ]:
# 合併訓練集與測試集，並利用part進行分辨
df_train['part'] = 'train'
df_test['part'] = 'test'

# 訓練集 + 測試集 合併
df_main = pd.concat([df_train, df_test], ignore_index=True, sort=False)
print(f"合併後總筆數: {df_main.shape}")

# 建立日期特徵 
df_main['month'] = df_main['date'].dt.month
df_main['day'] = df_main['date'].dt.day
df_main['dayofweek'] = df_main['date'].dt.dayofweek  # 0=Monday, 6=Sunday

# 建立發薪日特徵 (Payday Logic) 每個月的 15 號 OR 每個月的最後一天 (is_month_end)，再轉成 0/1 標籤
df_main['is_payday'] = (df_main['day'] == 15) | (df_main['date'].dt.is_month_end)
df_main['is_payday'] = df_main['is_payday'].astype(int)
print("日期與發薪日特徵已建立。")

# 排序順序：先分店 -> 再商品 -> 最後按日期
df_main = df_main.sort_values(['store_nbr', 'family', 'date'])
print("-" * 30)
display(df_main[df_main['part'] == 'test'].head()[['date', 'store_nbr', 'family']])

合併後總筆數: (3029400, 7)
日期與發薪日特徵已建立。
------------------------------


,date,store_nbr,family
3000888,2017-08-16,1,AUTOMOTIVE
3002670,2017-08-17,1,AUTOMOTIVE
3004452,2017-08-18,1,AUTOMOTIVE
3006234,2017-08-19,1,AUTOMOTIVE
3008016,2017-08-20,1,AUTOMOTIVE


In [ ]:
# 合併 Stores,Oil,Holidays
df_final = pd.merge(df_main, df_stores, on='store_nbr', how='left')
df_final = pd.merge(df_final, df_oil, on='date', how='left')
df_final = pd.merge(df_final, df_holidays_clean, on='date', how='left')

# 重新命名衝突的欄位
rename_dict = {
    'type_x': 'store_type',
    'type_y': 'holiday_type'
}
df_final = df_final.rename(columns=rename_dict)


#判斷假日的適用地區是否符合商店位置
is_national = df_final['locale'] == 'National'
is_regional = (df_final['locale'] == 'Regional') & (df_final['locale_name'] == df_final['state'])
is_local = (df_final['locale'] == 'Local') & (df_final['locale_name'] == df_final['city'])
# 只要符合其中一個，就是有效假日
mask_holiday_match = is_national | is_regional | is_local

# 要清除的欄位
cols_to_clear = ['holiday_type', 'locale', 'locale_name', 'description']
condition_to_clear = (~mask_holiday_match) & (df_final['holiday_type'].notna())
# 清除 (設為 NaN)
df_final.loc[condition_to_clear, cols_to_clear] = np.nan

# 去除重複值
df_final = df_final.drop_duplicates(subset=['date', 'store_nbr', 'family'])
print(f"合併前筆數: {df_main.shape[0]}")
print(f"合併後筆數: {df_final.shape[0]}")

# 移除 description 欄位
if 'description' in df_final.columns:
    df_final = df_final.drop(columns=['description'])
    print("已移除 description 欄位。")

# 油價補值 (後向填補)
df_final['dcoilwtico'] = df_final['dcoilwtico'].fillna(method='bfill')

# 假日相關補值
df_final['holiday_type'] = df_final['holiday_type'].fillna('WorkDay') # 沒放假就是工作日
df_final['locale'] = df_final['locale'].fillna('Normal')              # 沒有特定地區就是 Normal
# locale_name 也補 'Normal'
df_final['locale_name'] = df_final['locale_name'].fillna('Normal')
# 檢查一下是否還有 NaN 
print("locale_name 缺值數:", df_final['locale_name'].isna().sum())
display(df_final.head())

合併前筆數: 3029400
合併後筆數: 3029400
已移除 description 欄位。


C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_24660\1188102547.py:38: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_final['dcoilwtico'] = df_final['dcoilwtico'].fillna(method='bfill')


locale_name 缺值數: 0


,id,date,store_nbr,family,sales,onpromotion,part,month,day,dayofweek,is_payday,city,state,store_type,cluster,dcoilwtico,holiday_type,locale,locale_name
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train,1,1,1,0,Quito,Pichincha,D,13,93.140000,Holiday,National,Ecuador
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,train,1,2,2,0,Quito,Pichincha,D,13,93.140000,WorkDay,Normal,Normal
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,train,1,3,3,0,Quito,Pichincha,D,13,92.970000,WorkDay,Normal,Normal
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,train,1,4,4,0,Quito,Pichincha,D,13,93.120000,WorkDay,Normal,Normal
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,train,1,5,5,0,Quito,Pichincha,D,13,93.146667,WorkDay,Normal,Normal


In [ ]:
#定義 validation 的起始日期
val_start_date = '2017-04-15'
val_end_date = '2017-08-15'

# 基礎欄位
base_cols = [
    'id','date','part','dcoilwtico','locale_name','store_type','city',
    'dayofweek','sales','store_nbr','onpromotion','locale','holiday_type',
    'state','family'
]

model_df = df_final[base_cols].copy()

# 從 date 分出年份、月份、幾號
model_df['year'] = model_df['date'].dt.year
model_df['month'] = model_df['date'].dt.month
model_df['day'] = model_df['date'].dt.day

# 依照 store_nbr、family、date 排序 Train/val/test split
train_df = model_df[model_df['part'] == 'train'].sort_values(['store_nbr','family','date']).reset_index(drop=True)
test_df = model_df[model_df['part'] == 'test'].sort_values(['store_nbr','family','date']).reset_index(drop=True)

val_mask = (train_df['date'] >= val_start_date) & (train_df['date'] <= val_end_date)
val_df = train_df.loc[val_mask].copy()
train_df = train_df.loc[~val_mask].copy()

# 「類別特徵」來做編碼
CATEGORICAL_COLS = [
    'locale_name', 'store_type', 'city', 'dayofweek',
    'store_nbr', 'locale', 'holiday_type', 'state',
    'family', 'year', 'month', 'day'
]

for col in CATEGORICAL_COLS:
    # 用 train_df 的 mapping 來轉換三個 dataframe
    # 從 train_df[col] 取出所有不同的值，排序後依序編上整數 ID（0,1,2,...）
    mapping = {v: i for i, v in enumerate(sorted(train_df[col].unique()))}
    
    train_df[col] = train_df[col].map(mapping).astype(int)

    # 訓練期間沒出現過的類別 → 統一編碼成 -1
    val_df[col]   = val_df[col].map(mapping).fillna(-1).astype(int)
    test_df[col]  = test_df[col].map(mapping).fillna(-1).astype(int)

    # 整體 +1（讓 unknown 變 0，其他類別變 1~K）
    train_df[col] = train_df[col] + 1
    val_df[col]   = val_df[col] + 1
    test_df[col]  = test_df[col] + 1




In [9]:
NUMERIC_COLS = [
    'dcoilwtico','onpromotion',
]
scale_cols = NUMERIC_COLS
# 把數值特徵線性縮放到 [0, 1] 範圍
scaler = MinMaxScaler()
# 只用 訓練資料 的 dcoilwtico、onpromotion 來學習每一欄的 min、max
scaler.fit(train_df[scale_cols])
for df_ in [train_df, val_df, test_df]:
    df_.loc[:, scale_cols] = scaler.transform(df_[scale_cols])

feature_cols = NUMERIC_COLS + CATEGORICAL_COLS


print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
print(f"train rows: {train_df.shape[0]}, val rows: {val_df.shape[0]}, test rows: {test_df.shape[0]}")
print("TRAIN null count:", train_df.isna().sum().sum())
print("VAL null count:", val_df.isna().sum().sum())
print("TEST null count:", test_df.isna().sum().sum())
print("train_df year unique:", sorted(train_df['year'].unique()))
print("val_df   year unique:", sorted(val_df['year'].unique()))


display(train_df.head())


Feature columns (14): ['dcoilwtico', 'onpromotion', 'locale_name', 'store_type', 'city', 'dayofweek', 'store_nbr', 'locale', 'holiday_type', 'state', 'family', 'year', 'month', 'day']
train rows: 2781702, val rows: 219186, test rows: 28512
TRAIN null count: 0
VAL null count: 0
TEST null count: 28512
train_df year unique: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
val_df   year unique: [np.int64(5)]


C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_24660\1587015688.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_.loc[:, scale_cols] = scaler.transform(df_[scale_cols])
C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_24660\1587015688.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_.loc[:, scale_cols] = scaler.transform(df_[scale_cols])
C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_24660\1587015688.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast t

,id,date,part,dcoilwtico,locale_name,store_type,city,dayofweek,sales,store_nbr,onpromotion,locale,holiday_type,state,family,year,month,day
0,0,2013-01-01,train,0.792965,5,4,19,2,0.0,1,0.0,2,4,13,1,1,1,1
1,1782,2013-01-02,train,0.792965,17,4,19,3,2.0,1,0.0,3,6,13,1,1,1,2
2,3564,2013-01-03,train,0.790951,17,4,19,4,3.0,1,0.0,3,6,13,1,1,1,3
3,5346,2013-01-04,train,0.792728,17,4,19,5,3.0,1,0.0,3,6,13,1,1,1,4
4,7128,2013-01-05,train,0.793044,17,4,19,6,5.0,1,0.0,3,6,13,1,1,1,5


In [10]:
# Sequence builders
def build_sequences(df, feature_cols, target_col, lookback):
    X_list, y_list = [], []
    # 依據 (store_nbr, family) 分組
    for (_, _), g in df.groupby(['store_nbr', 'family'], sort=False):
        g_sorted = g.sort_values('date')
        # 取出這個 group 內，所有特徵欄位 + 目標欄位，變成一個 numpy 陣列
        arr = g_sorted[feature_cols + [target_col]].to_numpy()
        for i in range(len(g_sorted) - lookback):
            # 從第 i 列開始取 lookback 列，所有特徵欄位
            window = arr[i:i+lookback, :-1]
            # 目標是第 i+lookback 列的最後一欄
            target = arr[i+lookback, -1]
            #如果 target 或 window 是 NaN → 丟掉這個樣本
            if np.isnan(target) or np.any(np.isnan(window)):
                continue
            X_list.append(window)
            y_list.append(target)
    # 如果完全沒有任何有效的樣本（例如該 df 太短或全部 NaN），就回傳 shape 正確但空的陣列
    if not X_list:
        return np.empty((0, lookback, len(feature_cols))), np.array([])
    return np.stack(X_list), np.array(y_list)

def build_sequences_for_predict(df, feature_cols, lookback):
    X_list, id_list = [], []
    # 以 (store_nbr, family) 分組，並在每組內照 date 排序
    for (_, _), g in df.groupby(['store_nbr', 'family'], sort=False):
        g_sorted = g.sort_values('date')
        feats = g_sorted[feature_cols].to_numpy()
        ids = g_sorted['id'].to_numpy()
        parts = g_sorted['part'].astype(str).to_numpy() if 'part' in g_sorted.columns else None
        for i in range(len(g_sorted) - lookback):
            target_idx = i + lookback
            # 只允許「target 那一筆是 test」的樣本被保留
            if parts is not None and parts[target_idx] != 'test':
                continue
            window = feats[i:target_idx]
            if np.any(np.isnan(window)):
                continue
            X_list.append(window)
            id_list.append(ids[target_idx])
    # 若沒有任何合法序列 → 回傳空陣列
    if not X_list:
        return np.empty((0, lookback, len(feature_cols))), np.array([])
    return np.stack(X_list), np.array(id_list)


In [11]:

# Build sequences
lookback = 45
X_train, y_train = build_sequences(train_df, feature_cols, 'sales', lookback)
X_val,   y_val   = build_sequences(val_df,   feature_cols, 'sales', lookback)

# 先存一份「原始 y」，之後算 RMSLE / 最後訓練會用到
y_train_orig = y_train.copy()
y_val_orig   = y_val.copy()

# 將目標 sales 做 log1p 轉換（log(1+sales)），讓分佈比較平滑
y_train = np.log1p(y_train)
y_val   = np.log1p(y_val)

# 把 train_df + val_df + test_df 全部接在一起，形成完整的「歷史＋測試」時間序列。
all_history = pd.concat([train_df, val_df, test_df], ignore_index=True, sort=False)
all_history = all_history.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
all_history.loc[all_history['part'] == 'test', 'sales'] = np.nan
X_test, test_ids = build_sequences_for_predict(all_history, feature_cols, lookback)

# 三個 DataFrame 中的 part 欄位都刪掉
for df_ in [train_df, val_df, test_df]:
    if 'part' in df_.columns:
        df_.drop(columns=['part'], inplace=True)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print("y_train (log) describe:")
print(pd.Series(y_train).describe())

print("\ny_train_orig (raw) describe:")
print(pd.Series(y_train_orig).describe())

print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, test_ids: {test_ids.shape}")



X_train: (2701512, 45, 14), y_train: (2701512,)
y_train (log) describe:
count    2.701512e+06
mean     2.895797e+00
std      2.699602e+00
min      0.000000e+00
25%      0.000000e+00
50%      2.397895e+00
75%      5.267858e+00
max      1.173381e+01
dtype: float64

y_train_orig (raw) describe:
count    2.701512e+06
mean     3.527131e+02
std      1.090623e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+01
75%      1.930000e+02
max      1.247170e+05
dtype: float64
X_val: (138996, 45, 14), y_val: (138996,)
X_test: (28512, 45, 14), test_ids: (28512,)


In [12]:
cat_cardinalities = {}
for col in CATEGORICAL_COLS:
    # 統計這個欄位在 train 裡有多少不同的類別值 | + 1：通常是為了保留一個「unknown / unseen」類別的位置
    cat_cardinalities[col] = train_df[col].nunique() + 1

cat_cardinalities


{'locale_name': 25,
 'store_type': 6,
 'city': 23,
 'dayofweek': 8,
 'store_nbr': 55,
 'locale': 5,
 'holiday_type': 7,
 'state': 17,
 'family': 34,
 'year': 6,
 'month': 13,
 'day': 32}

In [ ]:
class EmbRNN(nn.Module):
    def __init__(
        self,
        lookback,
        feature_cols,
        numeric_cols,
        categorical_cols,
        cat_cardinalities,
        rnn_type='gru',
        units=128,
        dropout=0.1,
        bidirectional=False
    ):
        super().__init__()

        self.feature_cols = feature_cols
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols
        self.lookback = lookback
        self.hidden_size = units
        self.bidirectional = bidirectional

        # numeric 在 feature_cols 中的 index
        self.num_idx = [feature_cols.index(c) for c in numeric_cols]

        # 每個 categorical 的 index + embedding
        self.cat_idx = {col: feature_cols.index(col) for col in categorical_cols}
        self.emb_layers = nn.ModuleDict()
        emb_total_dim = 0

        for col in categorical_cols:
            vocab_size = int(cat_cardinalities[col])
            emb_dim = min(16, vocab_size // 2 + 1)
            self.emb_layers[col] = nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=emb_dim
            )
            emb_total_dim += emb_dim

        # numeric 維度
        num_dim = len(self.num_idx)

        # 總輸入維度 = numeric + 所有 embedding
        input_dim = num_dim + emb_total_dim

        # 選擇 RNN/LSTM/GRU
        rnn_map = {
            'rnn': nn.RNN,
            'lstm': nn.LSTM,
            'gru': nn.GRU
        }
        assert rnn_type in rnn_map, f"Unsupported rnn_type: {rnn_type}"
        rnn_cls = rnn_map[rnn_type]

        self.rnn_type = rnn_type
        self.rnn = rnn_cls(
            input_size=input_dim,
            hidden_size=units,
            batch_first=True,
            bidirectional=bidirectional
        )
        self.dropout = nn.Dropout(dropout)

        # 雙向時 hidden 大小會 ×2
        rnn_out_dim = units * (2 if bidirectional else 1)

        self.fc1 = nn.Linear(rnn_out_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc_out = nn.Linear(32, 1)

    def forward(self, x):
        """
        x: (batch_size, lookback, n_features)  # numpy 轉成的 torch.Tensor
        """
        # 先確保 x 要是 float32
        x = x.float()

        # numeric 特徵
        num_x = x[:, :, self.num_idx]  # (B, T, num_dim)

        # categorical embedding
        emb_list = []
        for col in self.categorical_cols:
            idx = self.cat_idx[col]
            # 取出這個欄位，轉成 long 當作 embedding index
            cat_ids = x[:, :, idx].long()  # (B, T)
            emb = self.emb_layers[col](cat_ids)  # (B, T, emb_dim)
            emb_list.append(emb)

        # 串接 numeric + 所有 embedding
        seq_repr = torch.cat([num_x] + emb_list, dim=-1)  # (B, T, input_dim)

        # RNN forward
        rnn_out, _ = self.rnn(seq_repr)  # (B, T, hidden*dir)
        # 取最後一個 time step 的輸出
        h_last = rnn_out[:, -1, :]  # (B, hidden*dir)

        h = self.dropout(h_last)
        h = F.relu(self.fc1(h))
        h = F.relu(self.fc2(h))
        out = self.fc_out(h)  # (B, 1)
        return out.squeeze(-1)  # (B,)


In [ ]:
def train_torch_model(
    model,
    X_train, y_train,
    X_val, y_val,
    lr=1e-3,
    batch_size=64,
    epochs=50,
    patience=3,
    lr_factor=0.5,
    lr_patience=2,
    min_lr=1e-5,
):
    """
    X_train, y_train, X_val, y_val: numpy arrays
        - X_* shape: (N, lookback, n_features)
        - y_* shape: (N,)  # log1p(sales)
    """
    model = model.to(device)

    # 建 DataLoader
    train_ds = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train.astype('float32'))
    )
    val_ds = TensorDataset(
        torch.from_numpy(X_val),
        torch.from_numpy(y_val.astype('float32'))
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    epochs_no_improve_lr = 0

    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0.0
        n_train = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)  # (B,)
            loss = criterion(preds, yb)
            loss.backward()

            # gradient clipping
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss_sum += loss.item() * xb.size(0)
            n_train += xb.size(0)

        train_loss = train_loss_sum / max(n_train, 1)

        # 驗證
        model.eval()
        val_loss_sum = 0.0
        n_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss_sum += loss.item() * xb.size(0)
                n_val += xb.size(0)

        val_loss = val_loss_sum / max(n_val, 1)

        # Early stopping & LR 調整
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = model.state_dict()
            epochs_no_improve = 0
            epochs_no_improve_lr = 0
        else:
            epochs_no_improve += 1
            epochs_no_improve_lr += 1

        # Reduce LR
        if epochs_no_improve_lr >= lr_patience:
            for g in optimizer.param_groups:
                new_lr = max(g['lr'] * lr_factor, min_lr)
                g['lr'] = new_lr
            epochs_no_improve_lr = 0

        # EarlyStopping
        if epochs_no_improve >= patience:
            break

    # 還原最佳權重
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_loss


In [ ]:
def rmsle(y_true, y_pred):
    # 把真實值 / 預測值都裁到 >= 0
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean(np.square(np.log1p(y_pred) - np.log1p(y_true))))

def run_embedding_bayes_for_model_torch(
    model_type,             # 'rnn' / 'lstm' / 'gru'
    X_train, y_train,       # log1p(y)
    X_val, y_val,           # log1p(y)
    y_val_orig,             # 原始 sales，用來算 RMSLE
    feature_cols,
    numeric_cols,
    categorical_cols,
    cat_cardinalities,
    lookback,
    n_calls=30,
    n_initial_points=10,
    random_state=42
):

    # 每輪 BO 開始先把舊的 CUDA fragment 清空
    torch.cuda.empty_cache()

    search_space = [
        Integer(64, 180, name='units'),
        Real(0.15, 0.35, name='dropout'),
        Categorical([32, 64], name='batch_size'),
        Integer(40, 120, name='epochs'),
        Real(2e-4, 1.5e-3, prior='log-uniform', name='lr'),
        Categorical([False, True], name='bidirectional'),
    ]

    results = []

    def objective(params):
        units, dropout, batch_size, epochs, lr, bidirectional = params

        cfg = {
            'units': int(units),
            'dropout': float(dropout),
            'batch_size': int(batch_size),
            'epochs': int(epochs),
            'lr': float(lr),
            'bidirectional': bool(bidirectional)
        }

        print(f"\n[BO-{model_type.upper()}] Training config = {cfg}")

        # ---- 建立模型 ----
        model = EmbRNN(
            lookback=lookback,
            feature_cols=feature_cols,
            numeric_cols=numeric_cols,
            categorical_cols=categorical_cols,
            cat_cardinalities=cat_cardinalities,
            rnn_type=model_type,
            units=cfg['units'],
            dropout=cfg['dropout'],
            bidirectional=cfg['bidirectional']
        )

        # ---- 訓練模型 ----
        model, best_val_loss = train_torch_model(
            model,
            X_train, y_train,
            X_val, y_val,
            lr=cfg['lr'],
            batch_size=cfg['batch_size'],
            epochs=cfg['epochs'],
            patience=3,
            lr_factor=0.5,
            lr_patience=2,
            min_lr=1e-5
        )

        # ---- 分批預測 validation ----
        model.eval()
        val_ds = TensorDataset(torch.from_numpy(X_val), torch.zeros(len(X_val)))
        val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)

        preds_log_list = []
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(device)
                preds_batch = model(xb)              # (B,)
                preds_log_list.append(preds_batch.cpu().numpy())

        preds_log_all = np.concatenate(preds_log_list, axis=0).squeeze()
        preds = np.expm1(preds_log_all)

        # ---- 算 RMSLE ----
        score = rmsle(y_val_orig, preds)

        results.append({
            'model_type': model_type,
            **cfg,
            'rmsle': float(score)
        })
        print(f"[BO-{model_type.upper()}] → RMSLE = {score:.4f}")

        # 釋放 GPU memory
        del model
        torch.cuda.empty_cache()

        return score

    bo_result = gp_minimize(
        func=objective,
        dimensions=search_space,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        random_state=random_state,
        verbose=False
    )

    res_df = pd.DataFrame(results).sort_values('rmsle')
    return res_df, bo_result


In [ ]:
# Bayesian Optimization
n_calls = 35          # 用跑35次
n_initial_points = 15 # 其中15次使用random


# 使用 numpy 隨機抽樣，限制訓練樣本數
max_train_samples = 200000
if len(X_train) > max_train_samples:
    idx = np.random.choice(len(X_train), max_train_samples, replace=False)
    X_train_use = X_train[idx]
    y_train_use = y_train[idx]
else:
    X_train_use, y_train_use = X_train, y_train


model_types = ['rnn', 'lstm', 'gru'] 

all_results = []   # 存每個模型所有嘗試的結果
summary_rows = []  # 存每個模型「最佳一組」的表

for m in model_types:
    print("\n" + "=" * 60)
    print(f"🔍 Start Bayesian Optimization for model_type = {m.upper()}")
    print("=" * 60)

    res_df, bo_res = run_embedding_bayes_for_model_torch(
        model_type=m,
        X_train=X_train_use,
        y_train=y_train_use,
        X_val=X_val,
        y_val=y_val,
        y_val_orig=y_val_orig,
        feature_cols=feature_cols,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        cat_cardinalities=cat_cardinalities,
        lookback=lookback,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        random_state=42
    )

    all_results.append(res_df)

    # 取出這個模型類型中 RMSLE 最小的一組
    best_row = res_df.iloc[0]
    summary_rows.append({
        'model_type': m,
        'best_rmsle': best_row['rmsle'],
        'best_units': best_row['units'],
        'best_dropout': best_row['dropout'],
        'best_batch_size': best_row['batch_size'],
        'best_epochs': best_row['epochs'],
        'best_lr': best_row['lr'],
        'best_bidirectional': best_row['bidirectional']
    })

# 把所有嘗試過的結果合併在一起
results_all = pd.concat(all_results, ignore_index=True).sort_values('rmsle')
print("\nTop 10 configs across ALL models:")
display(results_all.head(10))

# 每種類型的最佳結果 summary
summary_df = pd.DataFrame(summary_rows).sort_values('best_rmsle')
print("\nBest result for each model_type:")
display(summary_df)

# 找出最好的那一個模型 + 超參數
best_overall = summary_df.iloc[0]
print("\nOverall best model_type & config:")
print(best_overall)

# 轉成 best_cfg 結構
best_cfg = {
    'rnn_type': best_overall['model_type'],
    'units': int(best_overall['best_units']),
    'dropout': float(best_overall['best_dropout']),
    'batch_size': int(best_overall['best_batch_size']),
    'epochs': int(best_overall['best_epochs']),
    'lr': float(best_overall['best_lr']),
    'bidirectional': bool(best_overall['best_bidirectional'])
}
print("\nbest_cfg (for retraining on full data) =", best_cfg)



🔍 Start Bayesian Optimization for model_type = RNN

[BO-RNN] Training config = {'units': 156, 'dropout': 0.18668695797323276, 'batch_size': 64, 'epochs': 88, 'lr': 0.0004910898607972045, 'bidirectional': False}
[BO-RNN] → RMSLE = 0.8586

[BO-RNN] Training config = {'units': 117, 'dropout': 0.21674172222780436, 'batch_size': 32, 'epochs': 92, 'lr': 0.00022407509193348162, 'bidirectional': True}
[BO-RNN] → RMSLE = 0.8349

[BO-RNN] Training config = {'units': 173, 'dropout': 0.15015575316820287, 'batch_size': 64, 'epochs': 89, 'lr': 0.0006859050226136135, 'bidirectional': False}
[BO-RNN] → RMSLE = 0.8277

[BO-RNN] Training config = {'units': 67, 'dropout': 0.25495493205167785, 'batch_size': 32, 'epochs': 44, 'lr': 0.0014227406173477584, 'bidirectional': False}
[BO-RNN] → RMSLE = 0.8364

[BO-RNN] Training config = {'units': 75, 'dropout': 0.2736772018666175, 'batch_size': 32, 'epochs': 119, 'lr': 0.000512243106652762, 'bidirectional': True}
[BO-RNN] → RMSLE = 0.8065

[BO-RNN] Training con

,model_type,units,dropout,batch_size,epochs,lr,bidirectional,rmsle
70,gru,127,0.214303,64,67,0.000216,True,0.653779
35,lstm,113,0.349869,64,67,0.000590,False,0.767872
71,gru,150,0.210765,64,40,0.000214,True,0.772601
72,gru,117,0.216742,32,92,0.000224,True,0.778669
73,gru,173,0.150156,64,89,0.000686,False,0.779503
74,gru,64,0.218166,64,106,0.000212,True,0.783101
75,gru,149,0.207153,64,110,0.000200,True,0.786180
36,lstm,156,0.186687,64,88,0.000491,False,0.787705
37,lstm,143,0.159588,64,51,0.000285,False,0.791569
38,lstm,173,0.150156,64,89,0.000686,False,0.798470



Best result for each model_type:


,model_type,best_rmsle,best_units,best_dropout,best_batch_size,best_epochs,best_lr,best_bidirectional
2,gru,0.653779,127,0.214303,64,67,0.000216,True
1,lstm,0.767872,113,0.349869,64,67,0.000590,False
0,rnn,0.806279,103,0.335885,32,56,0.000450,True



Overall best model_type & config:
model_type                 gru
best_rmsle            0.653779
best_units                 127
best_dropout          0.214303
best_batch_size             64
best_epochs                 67
best_lr               0.000216
best_bidirectional        True
Name: 2, dtype: object

best_cfg (for retraining on full data) = {'rnn_type': 'gru', 'units': 127, 'dropout': 0.21430344793714823, 'batch_size': 64, 'epochs': 67, 'lr': 0.00021552392755073329, 'bidirectional': True}


In [ ]:
# 合併 train + val（y 是 log1p）
X_full = np.concatenate([X_train, X_val], axis=0)
y_full_log = np.concatenate([y_train, y_val], axis=0)

# 如果前面有舊的 best_model_torch，可以先刪掉釋放 GPU
try:
    del best_model_torch
except NameError:
    pass
torch.cuda.empty_cache()

# 建立最佳 PyTorch 模型
best_model_torch = EmbRNN(
    lookback=lookback,
    feature_cols=feature_cols,
    numeric_cols=NUMERIC_COLS,
    categorical_cols=CATEGORICAL_COLS,
    cat_cardinalities=cat_cardinalities,
    rnn_type=best_cfg['rnn_type'],
    units=int(best_cfg['units']),
    dropout=float(best_cfg['dropout']),
    bidirectional=bool(best_cfg['bidirectional'])
).to(device)

# 再訓練一次（用 full data）
best_model_torch, _ = train_torch_model(
    best_model_torch,
    X_full, y_full_log,
    X_val, y_val,  
    lr=float(best_cfg['lr']),
    batch_size=int(best_cfg['batch_size']),
    epochs=int(best_cfg['epochs']),
    patience=5,
    lr_factor=0.5,
    lr_patience=3,
    min_lr=1e-5
)

# 4. 預測 test（log1p → expm1）
best_model_torch.eval()

pred_batches = []
batch_size_pred = 512   

with torch.no_grad():
    for start in range(0, len(X_test), batch_size_pred):
        end = start + batch_size_pred
        x_batch = torch.from_numpy(X_test[start:end]).to(device)
        out = best_model_torch(x_batch)          # shape: (B, 1) 或 (B,)
        pred_batches.append(out.cpu().numpy())

test_preds_log_t = np.concatenate(pred_batches, axis=0).squeeze()
test_preds = np.expm1(test_preds_log_t)

submission = pd.DataFrame({
    'id': test_ids.astype(int),
    'sales': np.clip(test_preds, 0, None)
})

submission = submission.sort_values("id").reset_index(drop=True)

submission_path = 'submission_rnn_lstm_gru_torch.csv'
submission.to_csv(submission_path, index=False)
print(f"Saved submission to {submission_path} | rows: {len(submission)} | sales>=0: {(submission['sales']>=0).mean():.3f}")


Saved submission to submission_rnn_lstm_gru_torch.csv | rows: 28512 | sales>=0: 1.000
